# Pakistani Law RAG — LLM Generation + LLM-as-a-Judge Evaluation

This notebook:
1. Connects to the HF Inference API for LLM generation (Mistral-7B)
2. Builds the full RAG pipeline: retrieve → generate
3. Implements **Faithfulness** evaluation (claim extraction + verification)
4. Implements **Relevancy** evaluation (alternate query generation + cosine similarity)
5. Runs the ablation study across all configurations
6. Saves all results for the report

**Files to upload before running:**
- `chunks_fixed.json`
- `chunks_recursive.json`
- `retrieval.py`

**Runtime:** T4 GPU (for CrossEncoder in retrieval.py)

---
## Cell 1 — Install Dependencies

In [ ]:
!pip install -q pinecone sentence-transformers rank-bm25 huggingface_hub tqdm

---
## Cell 2 — Configuration (EDIT THIS CELL)

In [ ]:
# ============================================================
#  FILL IN YOUR OWN VALUES
# ============================================================

PINECONE_API_KEY = "YOUR_PINECONE_API_KEY_HERE"
PINECONE_INDEX   = "pakistani-law"
HF_TOKEN         = "YOUR_HUGGINGFACE_TOKEN_HERE"  # hf.co → Settings → Access Tokens → New token (read)

# LLM for generation and for judging
# These are free on HF Inference API — no credit card needed
GENERATION_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
JUDGE_MODEL      = "mistralai/Mistral-7B-Instruct-v0.3"  # same model is fine for judge

# Retrieval defaults for the main pipeline
DEFAULT_STRATEGY = "fixed"
DEFAULT_MODE     = "hybrid_rerank"
DEFAULT_TOP_K    = 5

print("Config loaded.")

---
## Cell 3 — Imports and Load Retrieval System

In [ ]:
import json
import re
import time
import torch
import numpy as np
from tqdm import tqdm
from huggingface_hub import InferenceClient
from sentence_transformers import SentenceTransformer

# Import our retrieval module (must be uploaded to this session)
from retrieval import load_retrieval_system, retrieve

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Load the full retrieval system
load_retrieval_system(
    pinecone_api_key       = PINECONE_API_KEY,
    index_name             = PINECONE_INDEX,
    fixed_chunks_path      = "chunks_fixed.json",
    recursive_chunks_path  = "chunks_recursive.json",
    device                 = device
)

print("Retrieval system loaded.")

---
## Cell 4 — HF Inference Client

The InferenceClient handles all API calls to HF-hosted LLMs.
You are NOT running the LLM locally — it runs on HF's servers.
Free tier allows ~1000 requests/day.

In [ ]:
client = InferenceClient(token=HF_TOKEN)

def call_llm(prompt: str,
             model: str = GENERATION_MODEL,
             max_new_tokens: int = 512,
             temperature: float = 0.1) -> str:
    """
    Call the HF Inference API and return the generated text.
    temperature=0.1 keeps outputs deterministic for evaluation.
    Retries once on failure.
    """
    for attempt in range(2):
        try:
            response = client.text_generation(
                prompt,
                model           = model,
                max_new_tokens  = max_new_tokens,
                temperature     = temperature,
                do_sample       = temperature > 0,
                return_full_text = False
            )
            return response.strip()
        except Exception as e:
            if attempt == 0:
                print(f"LLM call failed ({e}), retrying in 5s...")
                time.sleep(5)
            else:
                print(f"LLM call failed twice: {e}")
                return ""


# Sanity check — costs 1 API call
test = call_llm("[INST] Say hello in one sentence. [/INST]")
print(f"LLM test response: {test}")

---
## Cell 5 — RAG Generation Prompt

The generation prompt wraps the retrieved context and query.
Mistral uses the `[INST]...[/INST]` format.
We instruct the model to:
- Answer ONLY from the provided context
- Cite which source each claim comes from
- Say 'not found' if the answer is not in the context

Grounding the model to the context is critical for high faithfulness scores.

In [ ]:
def build_generation_prompt(query: str, context: str) -> str:
    return f"""[INST] You are a knowledgeable assistant specialising in Pakistani law.
Answer the question below using ONLY the information provided in the context.
Do not use any external knowledge. If the answer is not in the context, say:
"This information is not available in the provided legal documents."

Context:
{context}

Question: {query}

Provide a clear, accurate answer based strictly on the context above. [/INST]"""


def rag_generate(query: str,
                 chunk_strategy: str = DEFAULT_STRATEGY,
                 retrieval_mode: str = DEFAULT_MODE,
                 top_k: int = DEFAULT_TOP_K) -> dict:
    """
    Full RAG pipeline: retrieve + generate.
    Returns a dict with query, context, chunks, answer, and latency.
    """
    # Step 1: Retrieve
    t0 = time.time()
    retrieval_result = retrieve(query,
                                chunk_strategy=chunk_strategy,
                                retrieval_mode=retrieval_mode,
                                top_k=top_k)
    retrieval_time = round((time.time() - t0) * 1000, 1)

    # Step 2: Generate
    t0 = time.time()
    prompt = build_generation_prompt(query, retrieval_result['context'])
    answer = call_llm(prompt, max_new_tokens=512)
    generation_time = round((time.time() - t0) * 1000, 1)

    return {
        "query":           query,
        "answer":          answer,
        "context":         retrieval_result['context'],
        "chunks":          retrieval_result['chunks'],
        "chunk_strategy":  chunk_strategy,
        "retrieval_mode":  retrieval_mode,
        "retrieval_ms":    retrieval_time,
        "generation_ms":   generation_time,
        "total_ms":        retrieval_time + generation_time
    }


# Quick test
test_result = rag_generate("What is the punishment for murder in Pakistan?")
print(f"Query:  {test_result['query']}")
print(f"Answer: {test_result['answer'][:400]}")
print(f"Timing: retrieval={test_result['retrieval_ms']}ms  generation={test_result['generation_ms']}ms")

---
## Cell 6 — Faithfulness Evaluation: Claim Extraction

Step 1 of faithfulness: extract all factual claims from the generated answer.
We prompt the LLM to return claims as a numbered list, one per line.
Then parse the response into a Python list.

In [ ]:
def extract_claims(answer: str) -> list[str]:
    """
    Use the LLM to extract individual factual claims from an answer.
    Returns a list of claim strings.
    """
    prompt = f"""[INST] You are a precise claim extractor.
Extract every distinct factual claim from the following text.
Return ONLY a numbered list, one claim per line.
Each claim must be a complete, standalone sentence.
Do not include opinions, qualifications, or meta-commentary.

Text:
{answer}

Numbered list of factual claims: [/INST]"""

    response = call_llm(prompt, max_new_tokens=400, temperature=0.0)

    # Parse numbered list: "1. claim", "2. claim", etc.
    claims = []
    for line in response.strip().split('\n'):
        line = line.strip()
        # Match lines starting with a number and period/bracket
        match = re.match(r'^\d+[.)\s]+(.+)$', line)
        if match:
            claim = match.group(1).strip()
            if len(claim) > 10:  # filter out near-empty lines
                claims.append(claim)

    return claims


# Test claim extraction
test_claims = extract_claims(test_result['answer'])
print(f"Extracted {len(test_claims)} claims from answer:")
for i, claim in enumerate(test_claims):
    print(f"  {i+1}. {claim}")

---
## Cell 7 — Faithfulness Evaluation: Claim Verification

Step 2 of faithfulness: for each extracted claim, ask the LLM:
"Is this claim supported by the context? Yes or No."
Faithfulness Score = supported claims / total claims.

In [ ]:
def verify_claim(claim: str, context: str) -> dict:
    """
    Verify whether a single claim is supported by the context.
    Returns dict with 'supported' (bool) and 'reasoning' (str).
    """
    prompt = f"""[INST] You are a fact-checker for legal documents.
Determine if the following claim is supported by the provided context.
Answer with ONLY 'YES' or 'NO' on the first line, then one sentence of reasoning.

Context:
{context[:2000]}

Claim: {claim}

Is this claim supported by the context? [/INST]"""

    response = call_llm(prompt, max_new_tokens=100, temperature=0.0)

    first_line = response.strip().split('\n')[0].strip().upper()
    supported  = 'YES' in first_line and 'NO' not in first_line[:3]

    return {
        'claim':     claim,
        'supported': supported,
        'reasoning': response.strip()
    }


def compute_faithfulness(answer: str, context: str) -> dict:
    """
    Full faithfulness pipeline:
    1. Extract claims from answer
    2. Verify each claim against context
    3. Return score and detailed results
    """
    claims = extract_claims(answer)

    if not claims:
        return {
            'score':        0.0,
            'claims':       [],
            'n_claims':     0,
            'n_supported':  0
        }

    verifications = []
    for claim in claims:
        result = verify_claim(claim, context)
        verifications.append(result)
        time.sleep(0.5)  # small delay to avoid rate limiting

    n_supported = sum(1 for v in verifications if v['supported'])
    score       = n_supported / len(verifications)

    return {
        'score':        round(score, 4),
        'claims':       verifications,
        'n_claims':     len(verifications),
        'n_supported':  n_supported
    }


# Test faithfulness on our example
print("Running faithfulness evaluation on test answer...")
faith_result = compute_faithfulness(test_result['answer'], test_result['context'])

print(f"\nFaithfulness Score: {faith_result['score']:.2%}")
print(f"Claims: {faith_result['n_supported']}/{faith_result['n_claims']} supported")
print("\nClaim-by-claim:")
for v in faith_result['claims']:
    status = '✓' if v['supported'] else '✗'
    print(f"  {status} {v['claim'][:100]}")

---
## Cell 8 — Relevancy Evaluation: Alternate Query Generation + Cosine Similarity

Relevancy measures whether the answer actually addresses the question.
Method:
1. Given the generated answer, ask the LLM: what question does this answer?
   Generate 3 candidate questions.
2. Embed all 3 generated questions AND the original query with MiniLM.
3. Compute cosine similarity between each generated question and the original query.
4. Relevancy Score = mean of 3 similarities.

High score = the answer is addressing what was actually asked.

In [ ]:
# Load MiniLM for cosine similarity (already in memory from retrieval, but load fresh here)
similarity_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)


def generate_questions_from_answer(answer: str) -> list[str]:
    """
    Ask the LLM to generate 3 questions that the given answer responds to.
    Returns a list of up to 3 question strings.
    """
    prompt = f"""[INST] You are given an answer to a legal question about Pakistani law.
Generate exactly 3 different questions that this answer could be responding to.
Return ONLY the 3 questions as a numbered list, one per line. Nothing else.

Answer:
{answer}

3 questions this answer responds to: [/INST]"""

    response = call_llm(prompt, max_new_tokens=200, temperature=0.3)

    questions = []
    for line in response.strip().split('\n'):
        line = line.strip()
        match = re.match(r'^\d+[.)\s]+(.+\?)$', line)
        if match:
            q = match.group(1).strip()
            if len(q) > 10:
                questions.append(q)
        elif line.endswith('?') and len(line) > 10 and not line[0].isdigit():
            questions.append(line)

    return questions[:3]  # ensure max 3


def cosine_similarity(v1: np.ndarray, v2: np.ndarray) -> float:
    """Compute cosine similarity between two vectors."""
    v1_norm = v1 / (np.linalg.norm(v1) + 1e-10)
    v2_norm = v2 / (np.linalg.norm(v2) + 1e-10)
    return float(np.dot(v1_norm, v2_norm))


def compute_relevancy(query: str, answer: str) -> dict:
    """
    Full relevancy pipeline:
    1. Generate 3 questions from answer
    2. Compute cosine similarity between each and original query
    3. Return mean similarity as relevancy score
    """
    generated_questions = generate_questions_from_answer(answer)

    if not generated_questions:
        return {'score': 0.0, 'generated_questions': [], 'similarities': []}

    # Embed original query and generated questions
    query_vec = similarity_model.encode(query, convert_to_numpy=True)
    gen_vecs  = similarity_model.encode(generated_questions, convert_to_numpy=True)

    similarities = [
        round(cosine_similarity(query_vec, gen_vec), 4)
        for gen_vec in gen_vecs
    ]

    avg_score = round(float(np.mean(similarities)), 4)

    return {
        'score':               avg_score,
        'generated_questions': generated_questions,
        'similarities':        similarities
    }


# Test relevancy on our example
print("Running relevancy evaluation on test answer...")
relev_result = compute_relevancy(test_result['query'], test_result['answer'])

print(f"\nRelevancy Score: {relev_result['score']:.4f}")
print("Generated questions and similarities:")
for q, sim in zip(relev_result['generated_questions'], relev_result['similarities']):
    print(f"  [{sim:.4f}] {q}")
print(f"  Original query: {test_result['query']}")

---
## Cell 9 — Fixed Test Set (10-20 Queries)

This is your fixed evaluation set. Never change these queries —
all ablation comparisons must use the exact same queries.
Cover multiple acts and legal areas for a representative evaluation.

In [ ]:
TEST_SET = [
    # Pakistan Penal Code
    "What is the punishment for murder under the Pakistan Penal Code?",
    "How does the PPC define theft and what is its penalty?",
    "What constitutes defamation under Pakistani criminal law?",

    # Constitution
    "What fundamental rights does the Constitution of Pakistan guarantee?",
    "What are the qualifications required to become President of Pakistan?",

    # Anti-Terrorism Act
    "How does Pakistani law define a terrorist act?",
    "What are the special powers of courts under the Anti-Terrorism Act?",

    # Contract Act
    "What are the essential elements of a valid contract in Pakistan?",
    "When is a contract considered void under the Contract Act 1872?",

    # NAO
    "What offences fall under the National Accountability Ordinance?",

    # CrPC
    "What is the procedure for filing a First Information Report in Pakistan?",
    "What powers does a magistrate have under the Code of Criminal Procedure?",

    # Evidence
    "What types of evidence are admissible in Pakistani courts?",

    # Family Law
    "What are the legal requirements for divorce under Muslim Family Laws?",

    # Companies Act
    "What are the duties of directors under the Companies Act 2017?",
]

print(f"Fixed test set: {len(TEST_SET)} queries")
for i, q in enumerate(TEST_SET):
    print(f"  {i+1:2d}. {q}")

---
## Cell 10 — Full Evaluation Runner

Runs the complete RAG pipeline + evaluation on all test queries for one configuration.
This is the function you'll call for each ablation configuration.

In [ ]:
def run_evaluation(queries: list[str],
                   chunk_strategy: str = "fixed",
                   retrieval_mode: str = "hybrid_rerank",
                   top_k: int = 5,
                   config_name: str = "") -> dict:
    """
    Run full RAG + evaluation on a list of queries.
    Returns aggregated scores and per-query results.
    """
    config_name = config_name or f"{chunk_strategy}_{retrieval_mode}"
    print(f"\n{'='*60}")
    print(f"EVALUATING: {config_name}")
    print(f"Config: chunk_strategy={chunk_strategy}, retrieval_mode={retrieval_mode}")
    print(f"{'='*60}")

    results = []

    for i, query in enumerate(tqdm(queries, desc=config_name)):
        try:
            # 1. RAG: retrieve + generate
            rag = rag_generate(query,
                               chunk_strategy=chunk_strategy,
                               retrieval_mode=retrieval_mode,
                               top_k=top_k)
            time.sleep(1)  # rate limit buffer

            # 2. Faithfulness
            faith = compute_faithfulness(rag['answer'], rag['context'])
            time.sleep(1)

            # 3. Relevancy
            relev = compute_relevancy(query, rag['answer'])
            time.sleep(1)

            results.append({
                'query':              query,
                'answer':             rag['answer'],
                'context':            rag['context'],
                'faithfulness_score': faith['score'],
                'faithfulness_detail': faith,
                'relevancy_score':    relev['score'],
                'relevancy_detail':   relev,
                'retrieval_ms':       rag['retrieval_ms'],
                'generation_ms':      rag['generation_ms'],
                'total_ms':           rag['total_ms'],
                'chunk_strategy':     chunk_strategy,
                'retrieval_mode':     retrieval_mode
            })

            print(f"  Q{i+1:02d}: Faithfulness={faith['score']:.2%}  "
                  f"Relevancy={relev['score']:.4f}  "
                  f"Time={rag['total_ms']}ms")

        except Exception as e:
            print(f"  Q{i+1:02d}: ERROR — {e}")
            results.append({'query': query, 'error': str(e)})

    # Aggregate scores (skip errored queries)
    valid = [r for r in results if 'faithfulness_score' in r]

    agg = {
        'config_name':       config_name,
        'chunk_strategy':    chunk_strategy,
        'retrieval_mode':    retrieval_mode,
        'n_queries':         len(queries),
        'n_valid':           len(valid),
        'avg_faithfulness':  round(np.mean([r['faithfulness_score'] for r in valid]), 4) if valid else 0,
        'avg_relevancy':     round(np.mean([r['relevancy_score']    for r in valid]), 4) if valid else 0,
        'avg_retrieval_ms':  round(np.mean([r['retrieval_ms']       for r in valid]), 1) if valid else 0,
        'avg_generation_ms': round(np.mean([r['generation_ms']      for r in valid]), 1) if valid else 0,
        'avg_total_ms':      round(np.mean([r['total_ms']           for r in valid]), 1) if valid else 0,
        'per_query':         results
    }

    print(f"\nSUMMARY [{config_name}]")
    print(f"  Avg Faithfulness: {agg['avg_faithfulness']:.2%}")
    print(f"  Avg Relevancy:    {agg['avg_relevancy']:.4f}")
    print(f"  Avg Total Latency:{agg['avg_total_ms']}ms")

    return agg


print("Evaluation runner defined.")

---
## Cell 11 — Run Ablation Study

Evaluate all 4 configurations from the ablation table.
This will take ~30-40 minutes due to LLM API calls.
Results are saved after each config so you don't lose data if the session dies.

**Ablation configurations:**
| Config | Chunking | Retrieval |
|---|---|---|
| A | Fixed | Semantic only |
| B | Fixed | Hybrid + Rerank |
| C | Recursive | Semantic only |
| D | Recursive | Hybrid + Rerank |

In [ ]:
ABLATION_CONFIGS = [
    {"chunk_strategy": "fixed",     "retrieval_mode": "semantic_only",  "name": "A_fixed_semantic"},
    {"chunk_strategy": "fixed",     "retrieval_mode": "hybrid_rerank",  "name": "B_fixed_hybrid_rerank"},
    {"chunk_strategy": "recursive", "retrieval_mode": "semantic_only",  "name": "C_recursive_semantic"},
    {"chunk_strategy": "recursive", "retrieval_mode": "hybrid_rerank",  "name": "D_recursive_hybrid_rerank"},
]

all_ablation_results = {}

for cfg in ABLATION_CONFIGS:
    result = run_evaluation(
        queries        = TEST_SET,
        chunk_strategy = cfg["chunk_strategy"],
        retrieval_mode = cfg["retrieval_mode"],
        config_name    = cfg["name"]
    )
    all_ablation_results[cfg["name"]] = result

    # Save after each config — don't lose data
    with open(f"results_{cfg['name']}.json", "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    print(f"  Saved results_{cfg['name']}.json")

print("\n=== ALL ABLATION CONFIGS COMPLETE ===")

---
## Cell 12 — Print the Ablation Table

This is the exact table format for your report.

In [ ]:
print("\n" + "="*80)
print("ABLATION STUDY RESULTS")
print("="*80)
print(f"{'Config':<30} {'Chunking':<12} {'Retrieval':<20} {'Faith%':>8} {'Relev':>8} {'Latency(ms)':>12}")
print("-"*80)

for name, r in all_ablation_results.items():
    print(f"{r['config_name']:<30} "
          f"{r['chunk_strategy']:<12} "
          f"{r['retrieval_mode']:<20} "
          f"{r['avg_faithfulness']:>7.1%} "
          f"{r['avg_relevancy']:>8.4f} "
          f"{r['avg_total_ms']:>11.0f}")

print("="*80)

# Identify best config
best = max(all_ablation_results.values(),
           key=lambda r: r['avg_faithfulness'] + r['avg_relevancy'])
print(f"\nBest overall config: {best['config_name']}")
print(f"  Faithfulness: {best['avg_faithfulness']:.2%}")
print(f"  Relevancy:    {best['avg_relevancy']:.4f}")

---
## Cell 13 — Detailed Examples for Report (3 Required Queries)

The assignment requires showing claim extraction + verification
for at least 3 example queries. This cell formats them cleanly for copy-paste into your report.

In [ ]:
# Use results from best config
best_config_results = best['per_query']
valid_results = [r for r in best_config_results if 'faithfulness_score' in r]

# Pick 3 interesting ones — first, middle, last
example_indices = [0, len(valid_results)//2, len(valid_results)-1]
examples = [valid_results[i] for i in example_indices if i < len(valid_results)]

for ex_num, ex in enumerate(examples, 1):
    print(f"\n{'='*70}")
    print(f"EXAMPLE {ex_num} — Report Ready Output")
    print(f"{'='*70}")
    print(f"Query:    {ex['query']}")
    print(f"")
    print(f"Answer:")
    print(f"  {ex['answer'][:600]}")
    print(f"")
    print(f"Faithfulness Score: {ex['faithfulness_score']:.2%}")
    print(f"Claims extracted and verified:")
    for v in ex['faithfulness_detail']['claims']:
        status = 'SUPPORTED' if v['supported'] else 'NOT SUPPORTED'
        print(f"  [{status}] {v['claim']}")
    print(f"")
    print(f"Relevancy Score: {ex['relevancy_score']:.4f}")
    print(f"Generated questions from answer:")
    for q, sim in zip(ex['relevancy_detail']['generated_questions'],
                      ex['relevancy_detail']['similarities']):
        print(f"  [sim={sim:.4f}] {q}")

---
## Cell 14 — Save Everything and Download

In [ ]:
# Save master results file
with open("ablation_results_ALL.json", "w", encoding="utf-8") as f:
    json.dump(all_ablation_results, f, ensure_ascii=False, indent=2)

# Save clean summary as CSV for easy copy into report
csv_lines = ["Config,Chunking,Retrieval,Faithfulness,Relevancy,AvgLatencyMs"]
for name, r in all_ablation_results.items():
    csv_lines.append(
        f"{r['config_name']},{r['chunk_strategy']},{r['retrieval_mode']},"
        f"{r['avg_faithfulness']:.4f},{r['avg_relevancy']:.4f},{r['avg_total_ms']:.0f}"
    )
with open("ablation_summary.csv", "w") as f:
    f.write("\n".join(csv_lines))

print("Files saved. Downloading...")

from google.colab import files
files.download("ablation_results_ALL.json")
files.download("ablation_summary.csv")

---
## Cell 15 — Summary

In [ ]:
print("╔══════════════════════════════════════════════════╗")
print("║    LLM GENERATION + EVALUATION — COMPLETE       ║")
print("╠══════════════════════════════════════════════════╣")
print("║  ✓ HF Inference API connected (Mistral-7B)       ║")
print("║  ✓ RAG generation pipeline working               ║")
print("║  ✓ Faithfulness: claim extraction + verify       ║")
print("║  ✓ Relevancy: alt query gen + cosine sim         ║")
print(f"║  ✓ {len(TEST_SET)} test queries evaluated                   ║")
print("║  ✓ Ablation study across 4 configurations        ║")
print("║  ✓ Results saved as JSON + CSV                   ║")
print("╠══════════════════════════════════════════════════╣")
print("║  Next: Notebook 4 — Gradio App + HF Spaces       ║")
print("╚══════════════════════════════════════════════════╝")